In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "scikit-learn", "pandas", "seaborn",
                "matplotlib", "pillow"], check=False)

import os, warnings, json, heapq
from pathlib import Path
from collections import Counter
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc as sk_auc,
)
from sklearn.utils.class_weight import compute_class_weight
import tqdm as tqdm
try:
    import timm
    HAS_TIMM = True
    print(f"  timm {timm.__version__} loaded ")
except ImportError:
    HAS_TIMM = False
    raise RuntimeError("pip install timm")

CONFIG = {
    "metadata_path": "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/metadata.csv",
    "image_dirs": [
        "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/images/imgs_part_1/imgs_part_1",
        "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/images/imgs_part_2/imgs_part_2",
        "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/images/imgs_part_3/imgs_part_3",
    ],
    "save_dir": "/kaggle/working/",
    "class_names": ["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"],
    "num_classes":  6,
    "image_size":   300,
    "backbone":     "convnext_small.in12k_ft_in1k",
    "batch_size":   32,
    "num_epochs":   80,         
    "lr":           3e-5,
    "weight_decay": 5e-4,
    "dropout":      0.50,
    "accum_steps":  2,
    "warmup_epochs": 5,
    "swa_start":    18,          
    "swa_lr":       8e-6,        
    "mixup_alpha":      0.35,    
    "label_smoothing":  0.10,
    "focal_gamma":      1.0,
    
    "top_k": 5,      #for ensembling(5 checkpoints based on val f1) 
    "early_stop_patience": 25,   
    "val_split":  0.15,
    "test_split": 0.15,
    "seed":   42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "tta_n":  5,
}
BINARY_META = [
    "smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history",
    "has_piped_water", "has_sewage_system",
    "itch", "grew", "hurt", "changed", "bleed", "elevation",
]
CONTINUOUS_META = ["age", "fitspatrick", "diameter_1", "diameter_2"]
TOP_REGIONS = 15

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)

set_seed(CONFIG["seed"])
print(f"\n  Device : {CONFIG['device'].upper()}")
print(f"  Save   : {CONFIG['save_dir']}")
os.makedirs(CONFIG["save_dir"], exist_ok=True)

  timm 1.0.25 loaded 

  Device : CUDA
  Save   : /kaggle/working/


In [ ]:
def find_image(img_id, dirs):
    for d in dirs:
        p = Path(d) / img_id
        if p.exists(): return str(p)
    return None

def encode_region(df):
    if "region" not in df.columns: return df, []
    top = df["region"].value_counts().nlargest(TOP_REGIONS).index.tolist()
    df["region_clean"] = df["region"].where(df["region"].isin(top), other="other")
    dummies = pd.get_dummies(df["region_clean"], prefix="reg").astype(float)
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=["region_clean"], inplace=True)
    return df, dummies.columns.tolist()


def load_data(cfg):
    df = pd.read_csv(cfg["metadata_path"])
    print(f"\n{'─'*62}")
    print(f"  Metadata : {len(df)} rows | {df['patient_id'].nunique()} patients")

    cmap = {c: i for i, c in enumerate(cfg["class_names"])}
    df["label"] = df["diagnostic"].map(cmap)
    df.dropna(subset=["label"], inplace=True)
    df["label"] = df["label"].astype(int)

    df["img_path"] = df["img_id"].apply(lambda x: find_image(x, cfg["image_dirs"]))
    miss = df["img_path"].isna().sum()
    if miss: print(f"  [WARN] {miss} missing images dropped")
    df.dropna(subset=["img_path"], inplace=True)
    print(f"  Images   : {len(df)} found")

    bmap = {True:1,False:0,"True":1,"False":0,"true":1,"false":0,
            1:1,0:0,"YES":1,"NO":0,"yes":1,"no":0}
    for col in BINARY_META:
        if col in df.columns:
            df[col] = df[col].map(bmap).fillna(0).astype(float)

    if "gender" in df.columns:
        df["gender"] = df["gender"].map(
            {"MALE":0,"FEMALE":1,"male":0,"female":1}).fillna(0.5)

    for col in CONTINUOUS_META:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col].fillna(df[col].median(), inplace=True)

    df, region_cols = encode_region(df)

    scaler = StandardScaler()
    df[CONTINUOUS_META] = scaler.fit_transform(df[CONTINUOUS_META].values)

    base = ["age","gender"] + BINARY_META + CONTINUOUS_META
    base = [c for c in base if c in df.columns]
    meta_features = base + region_cols

    print(f"  Meta dim : {len(meta_features)} ({len(base)} base + {len(region_cols)} region-OHE)")
    print(f"  Classes  :")
    for cls, cnt in sorted(Counter(df["label"].values).items()):
        print(f"    {cfg['class_names'][cls]:5s}: {cnt}")
    print(f"{'─'*62}")
    return df, meta_features, scaler


def patient_split(df, cfg):
    pat = df.groupby("patient_id")["label"].agg(lambda x: x.mode()[0]).reset_index()
    tr_p, tmp_p = train_test_split(
        pat["patient_id"], test_size=cfg["val_split"]+cfg["test_split"],
        stratify=pat["label"], random_state=cfg["seed"])
    tmp_lbl = pat.loc[pat["patient_id"].isin(tmp_p), "label"]
    va_p, te_p = train_test_split(
        tmp_p, test_size=cfg["test_split"]/(cfg["val_split"]+cfg["test_split"]),
        stratify=tmp_lbl, random_state=cfg["seed"])
    tr = df[df["patient_id"].isin(tr_p)].reset_index(drop=True)
    va = df[df["patient_id"].isin(va_p)].reset_index(drop=True)
    te = df[df["patient_id"].isin(te_p)].reset_index(drop=True)
    print(f"\n  Patient-level split (no leakage):")
    print(f"    Train : {len(tr):4d} imgs | {tr['patient_id'].nunique()} patients")
    print(f"    Val   : {len(va):4d} imgs | {va['patient_id'].nunique()} patients")
    print(f"    Test  : {len(te):4d} imgs | {te['patient_id'].nunique()} patients")
    return tr, va, te


df, meta_features, scaler = load_data(CONFIG)
train_df, val_df, test_df = patient_split(df, CONFIG)
META_DIM = len(meta_features)
print(f"\n  META_DIM = {META_DIM}")


──────────────────────────────────────────────────────────────
  Metadata : 2298 rows | 1373 patients
  Images   : 2298 found
  Meta dim : 33 (19 base + 14 region-OHE)
  Classes  :
    ACK  : 730
    BCC  : 845
    MEL  : 52
    NEV  : 244
    SCC  : 192
    SEK  : 235
──────────────────────────────────────────────────────────────

  Patient-level split (no leakage):
    Train : 1608 imgs | 961 patients
    Val   :  351 imgs | 206 patients
    Test  :  339 imgs | 206 patients

  META_DIM = 33


In [ ]:
class PADDataset(Dataset):
    def __init__(self, df, meta_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.meta_cols = meta_cols
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["img_path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        meta  = torch.tensor(row[self.meta_cols].values.astype(np.float32))
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return img, meta, label


class CutOut:
    def __init__(self, size=56, p=0.40): self.size=size; self.p=p
    def __call__(self, img):
        if np.random.rand() > self.p: return img
        a = np.array(img); h,w = a.shape[:2]
        cy,cx = np.random.randint(h), np.random.randint(w)
        y1,y2 = max(0,cy-self.size//2), min(h,cy+self.size//2)
        x1,x2 = max(0,cx-self.size//2), min(w,cx+self.size//2)
        a[y1:y2,x1:x2] = 0
        return Image.fromarray(a)


def get_transforms(cfg, mode):
    sz   = cfg["image_size"]
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if mode == "train":
        return T.Compose([
            T.Resize((sz+40, sz+40)),
            T.RandomCrop(sz),
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.5),
            T.RandomRotation(30),
            T.ColorJitter(0.30, 0.30, 0.20, 0.05),
            T.RandomGrayscale(0.05),
            T.RandomPerspective(0.2, 0.3),
            T.RandAugment(num_ops=2, magnitude=9),
            CutOut(size=56, p=0.40),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])
    else: 
        return T.Compose([
            T.Resize((sz, sz)),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])


def make_sampler(df):
    counts  = Counter(df["label"].values)
    weights = [1.0 / counts[l] for l in df["label"].values]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


nw = min(4, os.cpu_count() or 1)
train_ds = PADDataset(train_df, meta_features, get_transforms(CONFIG, "train"))
val_ds   = PADDataset(val_df,   meta_features, get_transforms(CONFIG, "val"))
test_ds  = PADDataset(test_df,  meta_features, get_transforms(CONFIG, "val"))

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                          sampler=make_sampler(train_df),
                          num_workers=nw, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=nw, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=nw, pin_memory=True)

print(f"  Loaders ready | train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

  Loaders ready | train=1608 val=351 test=339


In [ ]:
class MetaBlock(nn.Module):
    def __init__(self, meta_dim, feat_dim):
        super().__init__()
        h = max(feat_dim // 4, 64)
        self.fc = nn.Sequential(
            nn.Linear(meta_dim, h), nn.ReLU(True), nn.Dropout(0.15),
            nn.Linear(h, feat_dim), nn.Sigmoid())
    def forward(self, feats, meta): return feats * self.fc(meta)


class MetaNet(nn.Module):
    def __init__(self, meta_dim, feat_dim):
        super().__init__()
        h = max(feat_dim // 4, 64)
        self.fc = nn.Sequential(
            nn.Linear(meta_dim, h), nn.ReLU(True), nn.Dropout(0.15),
            nn.Linear(h, feat_dim), nn.Tanh())
    def forward(self, feats, meta): return feats * (1.0 + self.fc(meta))


class SkinLesionNetV2(nn.Module):
    def __init__(self, num_classes, meta_dim, dropout=0.5,
                 backbone="convnext_small.in12k_ft_in1k"):
        super().__init__()
        self.backbone  = timm.create_model(backbone, pretrained=True,
                                           num_classes=0, global_pool="avg")
        self.feat_dim  = self.backbone.num_features      # 768
        self.meta_block = MetaBlock(meta_dim, self.feat_dim)
        self.meta_net   = MetaNet  (meta_dim, self.feat_dim)
        fused = self.feat_dim * 2                        # 1536

        self.head = nn.Sequential(
            nn.Linear(fused, 1024), nn.BatchNorm1d(1024), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(1024, 512),  nn.BatchNorm1d(512),  nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(512, num_classes),
        )

    def forward(self, img, meta):
        f   = self.backbone(img)
        mb  = self.meta_block(f, meta)
        mn  = self.meta_net  (f, meta)
        return self.head(torch.cat([mb, mn], dim=1))


def build_model(cfg, meta_dim):
    m = SkinLesionNetV2(cfg["num_classes"], meta_dim,
                        cfg["dropout"], cfg["backbone"]).to(cfg["device"])
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Model : {cfg['backbone']}")
    print(f"  Params: {n/1e6:.1f}M  feat_dim={m.feat_dim}  meta_dim={meta_dim}")
    return m

In [ ]:
from tqdm import tqdm

class LabelSmoothFocalLoss(nn.Module):
    def __init__(self, gamma=1.0, smoothing=0.1, weight=None):
        super().__init__()
        self.gamma     = gamma
        self.smoothing = smoothing
        self.weight    = weight

    def forward(self, logits, targets):
        n = logits.size(1)
        with torch.no_grad():
            soft = torch.full_like(logits, self.smoothing / n)
            soft.scatter_(1, targets.unsqueeze(1),
                          1.0 - self.smoothing + self.smoothing / n)
        log_p = torch.log_softmax(logits, 1)
        if self.weight is not None:
            loss = -(soft * log_p * self.weight[targets].unsqueeze(1)).sum(1)
        else:
            loss = -(soft * log_p).sum(1)
        pt = torch.exp(log_p).gather(1, targets.unsqueeze(1)).squeeze(1).detach()
        return ((1 - pt) ** self.gamma * loss).mean()


def get_class_weights(df, n, device):
    w = compute_class_weight("balanced", classes=np.arange(n), y=df["label"].values)
    return torch.tensor(w, dtype=torch.float32).to(device)

class WarmupCosine:
    def __init__(self, opt, warmup, total, base_lr, min_lr=1e-6):
        self.opt=opt; self.warmup=warmup; self.total=total
        self.base=base_lr; self.min=min_lr
    def step(self, ep):
        if ep <= self.warmup:
            lr = self.base * ep / self.warmup
        else:
            p = (ep - self.warmup) / (self.total - self.warmup)
            lr = self.min + 0.5*(self.base-self.min)*(1+np.cos(np.pi*p))
        for g in self.opt.param_groups: g["lr"] = lr
        return lr

def mixup(imgs, metas, labels, alpha):
    if alpha <= 0: return imgs, metas, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return (lam*imgs+(1-lam)*imgs[idx],
            lam*metas+(1-lam)*metas[idx],
            labels, labels[idx], lam)


def train_one_epoch(model, loader, optimizer, criterion, cfg, amp_scaler):
    model.train()
    device = cfg["device"]; alpha = cfg["mixup_alpha"]; accum = cfg["accum_steps"]
    run_loss=0.; preds_all=[]; lbls_all=[]
    optimizer.zero_grad()
    pbar = tqdm(loader, desc="Training", leave=False)
    for i, (imgs, meta, labels) in enumerate(pbar, 1):
        imgs, meta, labels = imgs.to(device), meta.to(device), labels.to(device)
        imgs, meta, ya, yb, lam = mixup(imgs, meta, labels, alpha)

        if amp_scaler:
            with torch.cuda.amp.autocast():
                logits = model(imgs, meta)
                loss   = lam*criterion(logits,ya) + (1-lam)*criterion(logits,yb)
            amp_scaler.scale(loss/accum).backward()
        else:
            logits = model(imgs, meta)
            loss   = lam*criterion(logits,ya) + (1-lam)*criterion(logits,yb)
            (loss/accum).backward()

        if i % accum == 0:
            if amp_scaler:
                amp_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                amp_scaler.step(optimizer); amp_scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad()

        run_loss += loss.item()*len(labels)
        preds_all.extend(logits.argmax(1).detach().cpu().numpy())
        lbls_all.extend(labels.cpu().numpy())
    

    return run_loss/len(loader.dataset), np.array(preds_all), np.array(lbls_all)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    run_loss=0.; preds_all=[]; lbls_all=[]; probs_all=[]
    pbar = tqdm(loader, desc="Validating", leave=False)
    for imgs, meta, labels in pbar:
        imgs, meta, labels = imgs.to(device), meta.to(device), labels.to(device)
        logits = model(imgs, meta)
        run_loss += criterion(logits, labels).item()*len(labels)
        probs = torch.softmax(logits, 1).cpu().numpy()
        probs_all.extend(probs)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        lbls_all.extend(labels.cpu().numpy())

    return (run_loss/len(loader.dataset),
            np.array(preds_all), np.array(lbls_all), np.array(probs_all))


@torch.no_grad()
def evaluate_tta(model, loader, cfg):
    model.eval()
    device = cfg["device"]
    preds_all=[]; lbls_all=[]; probs_all=[]
    for imgs, meta, labels in loader:
        imgs, meta, labels = imgs.to(device), meta.to(device), labels.to(device)
        acc = torch.zeros(imgs.size(0), cfg["num_classes"], device=device)
        acc += torch.softmax(model(imgs, meta), 1)
        acc += torch.softmax(model(TF.hflip(imgs), meta), 1)
        acc += torch.softmax(model(TF.vflip(imgs), meta), 1)
        acc += torch.softmax(model(TF.hflip(TF.vflip(imgs)), meta), 1)
        acc += torch.softmax(model(torch.rot90(imgs, k=1, dims=[2,3]), meta), 1)
        p = (acc / 5).cpu().numpy()
        probs_all.extend(p)
        preds_all.extend(np.argmax(p,1))
        lbls_all.extend(labels.cpu().numpy())
    return np.array(preds_all), np.array(lbls_all), np.array(probs_all)


def metrics_from(preds, labels, n):
    acc  = accuracy_score(labels, preds)
    bal  = balanced_accuracy_score(labels, preds)
    f1m  = f1_score(labels, preds, average="macro",  zero_division=0)
    f1pc = f1_score(labels, preds, average=None, labels=list(range(n)), zero_division=0)
    return acc, bal, f1m, f1pc


def print_ep(ep, phase, loss, acc, bal, f1, f1pc, cls):
    tag = " TRAIN" if "TRAIN" in phase else " VAL  "
    print(f"  [Epoch {ep:03d}] {tag}")
    print(f"  Loss={loss:.4f}  Acc={acc:.4f}  BalAcc={bal:.4f}  F1={f1:.4f}")
    print("  Per-class: " + "  ".join(f"{c}={v:.3f}" for c,v in zip(cls,f1pc)))


def save_cm(lbl, pred, cls, ep, phase, d):
    cm = confusion_matrix(lbl, pred)
    fig,ax = plt.subplots(figsize=(9,7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=cls, yticklabels=cls, ax=ax, linewidths=0.5)
    ax.set(xlabel="Predicted", ylabel="True",
           title=f"Confusion Matrix — {phase} @ Epoch {ep}")
    plt.tight_layout()
    p=f"{d}cm_{phase.lower()}_ep{ep:03d}.png"
    plt.savefig(p,dpi=130); plt.close(); print(f"  >> CM → {p}")


def print_report_cm(lbl, pred, cls, ep, phase, d):
    # print(f"\n{'═'*65}")
    print(f"  Classification Report  [{phase} | Epoch {ep}]")
    # print("═"*65)
    print(classification_report(lbl, pred, target_names=cls, zero_division=0))
    save_cm(lbl, pred, cls, ep, phase, d)

class TopKCheckpoints:
    def __init__(self, k, save_dir):
        self.k = k; self.d = save_dir; self.heap = []

    def maybe_save(self, score, epoch, model):
        path = f"{self.d}topk_ep{epoch:03d}_f1{score:.4f}.pth"
        if len(self.heap) < self.k:
            torch.save(model.state_dict(), path)
            heapq.heappush(self.heap, (score, path))
            print(f"  → Top-{self.k} checkpoint #{len(self.heap)} saved  f1={score:.4f}")
            return True
        if score > self.heap[0][0]:
            _, old = heapq.heappop(self.heap)
            if os.path.exists(old): os.remove(old)
            torch.save(model.state_dict(), path)
            heapq.heappush(self.heap, (score, path))
            print(f"  → Top-{self.k} updated  f1={score:.4f}  (evicted {os.path.basename(old)})")
            return True
        return False

    def paths_scores(self):
        return sorted(self.heap, key=lambda x: -x[0])

    def __len__(self): return len(self.heap)

In [ ]:
model = build_model(CONFIG, META_DIM)

class_w   = get_class_weights(train_df, CONFIG["num_classes"], CONFIG["device"])
criterion = LabelSmoothFocalLoss(
    gamma=CONFIG["focal_gamma"], smoothing=CONFIG["label_smoothing"], weight=class_w)

optimizer = optim.AdamW(model.parameters(),
                        lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
lr_sched  = WarmupCosine(optimizer, CONFIG["warmup_epochs"],
                         CONFIG["num_epochs"], CONFIG["lr"])
amp_scaler = torch.cuda.amp.GradScaler() if CONFIG["device"]=="cuda" else None

swa_model  = AveragedModel(model).to(CONFIG["device"])
swa_active = False

history = {k:[] for k in ["train_loss","val_loss","train_acc","val_acc",
                           "train_bal","val_bal","train_f1","val_f1"]}
best_val_f1     = 0.0
best_epoch      = 0
patience_ctr    = 0
topk_ckpts      = TopKCheckpoints(CONFIG["top_k"], CONFIG["save_dir"])

print(f"\n  ═══ Training (max {CONFIG['num_epochs']} ep | "
      f"patience={CONFIG['early_stop_patience']} | "
      f"SWA from ep {CONFIG['swa_start']}) ═══")

for ep in range(1, CONFIG["num_epochs"]+1):

    if not swa_active:
        lr_sched.step(ep)
    else:
        for g in optimizer.param_groups: g["lr"] = CONFIG["swa_lr"]

    tr_loss, tr_pred, tr_lbl = train_one_epoch(
        model, train_loader, optimizer, criterion, CONFIG, amp_scaler)
    tr_acc, tr_bal, tr_f1, tr_f1pc = metrics_from(tr_pred, tr_lbl, CONFIG["num_classes"])

    va_loss, va_pred, va_lbl, va_probs = evaluate(
        model, val_loader, criterion, CONFIG["device"])
    va_acc, va_bal, va_f1, va_f1pc = metrics_from(va_pred, va_lbl, CONFIG["num_classes"])

    if ep >= CONFIG["swa_start"]:
        if not swa_active:
            swa_active = True
            print(f"\n   SWA activated at epoch {ep}")
        swa_model.update_parameters(model)

    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(va_acc)
    history["train_bal"].append(tr_bal);   history["val_bal"].append(va_bal)
    history["train_f1"].append(tr_f1);     history["val_f1"].append(va_f1)

    print_ep(ep,"TRAIN",tr_loss,tr_acc,tr_bal,tr_f1,tr_f1pc,CONFIG["class_names"])
    print_ep(ep,"VAL",  va_loss,va_acc,va_bal,va_f1,va_f1pc,CONFIG["class_names"])

    if ep % 5 == 0:
        print_report_cm(va_lbl, va_pred, CONFIG["class_names"],
                        ep, "Val", CONFIG["save_dir"])

    topk_ckpts.maybe_save(va_f1, ep, model)

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1; best_epoch = ep; patience_ctr = 0
        torch.save({"epoch":ep, "model_state_dict":model.state_dict(),
                    "val_f1":va_f1, "val_bal":va_bal},
                   f"{CONFIG['save_dir']}best_model.pth")
        print(f"\n   BEST  val_f1={va_f1:.4f}  val_bal={va_bal:.4f}  (ep {ep})")
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG["early_stop_patience"]:
            print(f"\n   Early stopping at epoch {ep} "
                  f"(no improvement for {CONFIG['early_stop_patience']} epochs)")
            print(f"     Best was ep {best_epoch}  val_f1={best_val_f1:.4f}")
            break

print(f"\n  Training complete. Best ep={best_epoch}, val_f1={best_val_f1:.4f}")
print(f"  Top-{CONFIG['top_k']} checkpoints:")
for sc, path in topk_ckpts.paths_scores():
    print(f"    f1={sc:.4f}  {os.path.basename(path)}")

with open(f"{CONFIG['save_dir']}history.json","w") as f:
    json.dump(history, f)

  Model : convnext_small.in12k_ft_in1k
  Params: 51.9M  feat_dim=768  meta_dim=33

  ═══ Training (max 80 ep | patience=25 | SWA from ep 18) ═══


  [Epoch 001]  TRAIN
  Loss=3.5034  Acc=0.1872  BalAcc=0.1823  F1=0.1649
  Per-class: ACK=0.208  BCC=0.087  MEL=0.199  NEV=0.207  SCC=0.244  SEK=0.044
  [Epoch 001]  VAL  
  Loss=1.5104  Acc=0.1481  BalAcc=0.3050  F1=0.1471
  Per-class: ACK=0.047  BCC=0.047  MEL=0.227  NEV=0.330  SCC=0.195  SEK=0.036
  → Top-5 checkpoint #1 saved  f1=0.1471

   BEST  val_f1=0.1471  val_bal=0.3050  (ep 1)


  [Epoch 002]  TRAIN
  Loss=2.6086  Acc=0.2444  BalAcc=0.2496  F1=0.2055
  Per-class: ACK=0.200  BCC=0.077  MEL=0.391  NEV=0.184  SCC=0.320  SEK=0.061
  [Epoch 002]  VAL  
  Loss=1.2730  Acc=0.2165  BalAcc=0.4171  F1=0.2350
  Per-class: ACK=0.277  BCC=0.016  MEL=0.327  NEV=0.492  SCC=0.216  SEK=0.082
  → Top-5 checkpoint #2 saved  f1=0.2350

   BEST  val_f1=0.2350  val_bal=0.4171  (ep 2)


  [Epoch 003]  TRAIN
  Loss=1.7743  Acc=0.2624  BalAcc=0.2725  F1=0.2304
  Per-class: ACK=0.205  BCC=0.078  MEL=0.381  NEV=0.274  SCC=0.308  SEK=0.137
  [Epoch 003]  VAL  
  Loss=1.0488  Acc=0.3219  BalAcc=0.5274  F1=0.4218
  Per-class: ACK=0.288  BCC=0.092  MEL=0.636  NEV=0.731  SCC=0.220  SEK=0.562
  → Top-5 checkpoint #3 saved  f1=0.4218

   BEST  val_f1=0.4218  val_bal=0.5274  (ep 3)


  [Epoch 004]  TRAIN
  Loss=1.6229  Acc=0.3812  BalAcc=0.3829  F1=0.3394
  Per-class: ACK=0.291  BCC=0.033  MEL=0.542  NEV=0.474  SCC=0.393  SEK=0.303
  [Epoch 004]  VAL  
  Loss=0.8746  Acc=0.4274  BalAcc=0.6045  F1=0.4610
  Per-class: ACK=0.568  BCC=0.034  MEL=0.500  NEV=0.701  SCC=0.273  SEK=0.690
  → Top-5 checkpoint #4 saved  f1=0.4610

   BEST  val_f1=0.4610  val_bal=0.6045  (ep 4)


...

  [Epoch 054]  TRAIN
  Loss=0.9002  Acc=0.4988  BalAcc=0.4956  F1=0.4925
  Per-class: ACK=0.443  BCC=0.422  MEL=0.577  NEV=0.529  SCC=0.456  SEK=0.528
  [Epoch 054]  VAL  
  Loss=0.7271  Acc=0.7778  BalAcc=0.7339  F1=0.7168
  Per-class: ACK=0.831  BCC=0.810  MEL=0.636  NEV=0.800  SCC=0.471  SEK=0.753


  [Epoch 055]  TRAIN
  Loss=0.6268  Acc=0.6107  BalAcc=0.6116  F1=0.6093
  Per-class: ACK=0.593  BCC=0.561  MEL=0.636  NEV=0.649  SCC=0.568  SEK=0.649
  [Epoch 055]  VAL  
  Loss=0.7275  Acc=0.7892  BalAcc=0.7151  F1=0.7256
  Per-class: ACK=0.844  BCC=0.803  MEL=0.706  NEV=0.837  SCC=0.364  SEK=0.800
  Classification Report  [Val | Epoch 55]
              precision    recall  f1-score   support

         ACK       0.89      0.80      0.84       111
         BCC       0.73      0.89      0.80       114
         MEL       0.75      0.67      0.71         9
         NEV       0.80      0.88      0.84        41
         SCC       0.57      0.27      0.36        30
         SEK       0.82      0.78      0.80        46

    accuracy                           0.79       351
   macro avg       0.76      0.72      0.73       351
weighted avg       0.79      0.79      0.78       351

  >> CM → /kaggle/working/cm_val_ep055.png


  [Epoch 056]  TRAIN
  Loss=0.7650  Acc=0.5361  BalAcc=0.5340  F1=0.5317
  Per-class: ACK=0.473  BCC=0.495  MEL=0.563  NEV=0.576  SCC=0.515  SEK=0.568
  [Epoch 056]  VAL  
  Loss=0.6921  Acc=0.7806  BalAcc=0.7393  F1=0.7229
  Per-class: ACK=0.842  BCC=0.798  MEL=0.636  NEV=0.824  SCC=0.414  SEK=0.824


  [Epoch 057]  TRAIN
  Loss=0.8095  Acc=0.6070  BalAcc=0.6053  F1=0.6029
  Per-class: ACK=0.583  BCC=0.520  MEL=0.654  NEV=0.657  SCC=0.580  SEK=0.624
  [Epoch 057]  VAL  
  Loss=0.6670  Acc=0.7778  BalAcc=0.7266  F1=0.7089
  Per-class: ACK=0.842  BCC=0.802  MEL=0.545  NEV=0.829  SCC=0.456  SEK=0.779


  [Epoch 058]  TRAIN
  Loss=0.7223  Acc=0.6443  BalAcc=0.6391  F1=0.6365
  Per-class: ACK=0.595  BCC=0.532  MEL=0.715  NEV=0.684  SCC=0.662  SEK=0.632
  [Epoch 058]  VAL  
  Loss=0.6544  Acc=0.7806  BalAcc=0.7322  F1=0.7260
  Per-class: ACK=0.846  BCC=0.798  MEL=0.632  NEV=0.829  SCC=0.438  SEK=0.813

   Early stopping at epoch 58 (no improvement for 25 epochs)
     Best was ep 33  val_f1=0.7462

  Training complete. Best ep=33, val_f1=0.7462
  Top-5 checkpoints:
    f1=0.7462  topk_ep033_f10.7462.pth
    f1=0.7311  topk_ep036_f10.7311.pth
    f1=0.7300  topk_ep027_f10.7300.pth
    f1=0.7282  topk_ep035_f10.7282.pth
    f1=0.7271  topk_ep046_f10.7271.pth


In [ ]:
def custom_update_bn(loader, swa_model, device):
    swa_model.train()
    momenta = {}
    for m in swa_model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.running_mean.zero_()
            m.running_var.fill_(1.0)
            momenta[m] = m.momentum
            m.momentum  = None             
            m.num_batches_tracked.zero_()

    print(f"  Resetting {len(momenta)} BN layers and accumulating stats …")

    with torch.no_grad():
        for imgs, meta, _ in loader:
            imgs, meta = imgs.to(device), meta.to(device)
            swa_model(imgs, meta)   

    for m, mom in momenta.items():
        m.momentum = mom

    print(f"  BN statistics updated ")


if swa_active:
    print("\n  ── SWA finalization ──")
    custom_update_bn(train_loader, swa_model, CONFIG["device"])
    torch.save(swa_model.state_dict(),
               f"{CONFIG['save_dir']}swa_model.pth")
    print("  SWA model saved ")

    print("\n  SWA quick val check …")
    va_loss_swa, va_pred_swa, va_lbl_swa, _ = evaluate(
        swa_model, val_loader, criterion, CONFIG["device"])
    _,_,va_f1_swa,va_f1pc_swa = metrics_from(va_pred_swa, va_lbl_swa, CONFIG["num_classes"])
    print(f"  SWA val F1-macro = {va_f1_swa:.4f} "
          f"(best single = {best_val_f1:.4f})")
    print("  Per-class: " + "  ".join(
        f"{c}={v:.3f}" for c,v in zip(CONFIG["class_names"], va_f1pc_swa)))
else:
    print("  SWA was not activated (training stopped before swa_start)")




  ── SWA finalization ──
  Resetting 2 BN layers and accumulating stats …
  BN statistics updated 
  SWA model saved 

  SWA quick val check …


  SWA val F1-macro = 0.7139 (best single = 0.7462)
  Per-class: ACK=0.841  BCC=0.803  MEL=0.571  NEV=0.815  SCC=0.444  SEK=0.809


In [ ]:
print("\n  ══ Building Top-K Ensemble ══")
print(f"  Loading {len(topk_ckpts)} checkpoints …")

_tmp_model = build_model(CONFIG, META_DIM)
device     = CONFIG["device"]

ensemble_val_probs  = []
ensemble_test_probs = []

for rank, (sc, path) in enumerate(topk_ckpts.paths_scores(), 1):
    print(f"\n  [{rank}/{len(topk_ckpts)}] f1={sc:.4f}  {os.path.basename(path)}")
    state = torch.load(path, map_location=device)
    _tmp_model.load_state_dict(state)
    _tmp_model.to(device)

    _, va_lbl_k, va_probs_k = evaluate_tta(_tmp_model, val_loader, CONFIG)
    ensemble_val_probs.append(va_probs_k)

    _, te_lbl_k, te_probs_k = evaluate_tta(_tmp_model, test_loader, CONFIG)
    ensemble_test_probs.append(te_probs_k)

    acc,bal,f1m,f1pc = metrics_from(np.argmax(va_probs_k,1), va_lbl_k, CONFIG["num_classes"])
    print(f"    Val F1={f1m:.4f}  BalAcc={bal:.4f}")
    _tmp_model.cpu()  

if swa_active:
    print(f"\n  [{len(topk_ckpts)+1}/{len(topk_ckpts)+1}] Adding SWA model …")
    swa_state = torch.load(f"{CONFIG['save_dir']}swa_model.pth", map_location=device)
    _tmp_model.to(device)

    swa_eval  = AveragedModel(build_model(CONFIG, META_DIM)).to(device)
    swa_eval.load_state_dict(swa_state)

    _, _,   sva_probs = evaluate_tta(swa_eval, val_loader,  CONFIG)
    _, _,   ste_probs = evaluate_tta(swa_eval, test_loader, CONFIG)
    ensemble_val_probs.append(sva_probs)
    ensemble_test_probs.append(ste_probs)
    acc,bal,f1m,f1pc = metrics_from(np.argmax(sva_probs,1), va_lbl_k, CONFIG["num_classes"])
    print(f"    SWA Val F1={f1m:.4f}  BalAcc={bal:.4f}")
    swa_eval.cpu()

avg_val_probs  = np.mean(ensemble_val_probs,  axis=0)   
avg_test_probs = np.mean(ensemble_test_probs, axis=0)   
val_labels_np  = va_lbl_k   
test_labels_np = te_lbl_k

print(f"\n  ── Ensemble (uniform average) before threshold tuning ──")
ens_val_preds  = np.argmax(avg_val_probs,  axis=1)
acc,bal,f1m,f1pc = metrics_from(ens_val_preds, val_labels_np, CONFIG["num_classes"])
print(f"  Val  F1={f1m:.4f}  BalAcc={bal:.4f}")
print("  Per-class: " + "  ".join(
    f"{c}={v:.3f}" for c,v in zip(CONFIG["class_names"],f1pc)))

ens_test_preds = np.argmax(avg_test_probs, axis=1)
acc,bal,f1m,f1pc = metrics_from(ens_test_preds, test_labels_np, CONFIG["num_classes"])
print(f"  Test F1={f1m:.4f}  BalAcc={bal:.4f}")
print("  Per-class: " + "  ".join(
    f"{c}={v:.3f}" for c,v in zip(CONFIG["class_names"],f1pc)))


def tune_thresholds(val_probs, val_labels, n_classes, n_iter=6, n_steps=40):
    thresholds = np.ones(n_classes, dtype=float)

    def predict(thr):
        adj = val_probs / thr[np.newaxis,:]   # (N, C)
        return np.argmax(adj, axis=1)

    best_f1 = f1_score(val_labels, predict(thresholds),
                        average="macro", zero_division=0)
    print(f"\n  Threshold tuning  (baseline macro F1 = {best_f1:.4f})")

    for it in range(n_iter):
        improved = False
        for c in range(n_classes):
            t_c_star = thresholds[c]
            for t in np.linspace(0.20, 3.0, n_steps):
                thresholds[c] = t
                f1 = f1_score(val_labels, predict(thresholds),
                               average="macro", zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1; t_c_star = t; improved = True
            thresholds[c] = t_c_star
        if not improved:
            print(f"    Converged after iteration {it+1}")
            break

    print(f"  Tuned macro F1 on val = {best_f1:.4f}")
    print("  Thresholds: " + "  ".join(
        f"{c}={t:.3f}" for c,t in zip(CONFIG["class_names"], thresholds)))
    return thresholds, best_f1


print("\n  ── Per-class threshold tuning on val ensemble probs ──")
tuned_thresholds, tuned_val_f1 = tune_thresholds(
    avg_val_probs, val_labels_np, CONFIG["num_classes"])

val_preds_tuned = np.argmax(avg_val_probs / tuned_thresholds, axis=1)
acc,bal,f1m,f1pc = metrics_from(val_preds_tuned, val_labels_np, CONFIG["num_classes"])
print(f"\n  Val (ensemble + tuned thresholds): F1={f1m:.4f}  BalAcc={bal:.4f}")
print("  Per-class: " + "  ".join(
    f"{c}={v:.3f}" for c,v in zip(CONFIG["class_names"],f1pc)))

np.save(f"{CONFIG['save_dir']}tuned_thresholds.npy", tuned_thresholds)
print(f"  Thresholds saved ")


  ══ Building Top-K Ensemble ══
  Loading 5 checkpoints …


  Model : convnext_small.in12k_ft_in1k
  Params: 51.9M  feat_dim=768  meta_dim=33

  [1/5] f1=0.7462  topk_ep033_f10.7462.pth
    Val F1=0.7462  BalAcc=0.7678

  [2/5] f1=0.7311  topk_ep036_f10.7311.pth
    Val F1=0.7139  BalAcc=0.7339

  [3/5] f1=0.7300  topk_ep027_f10.7300.pth
    Val F1=0.7132  BalAcc=0.7362

  [4/5] f1=0.7282  topk_ep035_f10.7282.pth
    Val F1=0.7302  BalAcc=0.7476

  [5/5] f1=0.7271  topk_ep046_f10.7271.pth
    Val F1=0.7214  BalAcc=0.7481

  [6/6] Adding SWA model …
  Model : convnext_small.in12k_ft_in1k
  Params: 51.9M  feat_dim=768  meta_dim=33
    SWA Val F1=0.7341  BalAcc=0.7761

  ── Ensemble (uniform average) before threshold tuning ──
  Val  F1=0.7231  BalAcc=0.7449
  Per-class: ACK=0.853  BCC=0.826  MEL=0.500  NEV=0.825  SCC=0.475  SEK=0.860
  Test F1=0.7171  BalAcc=0.7643
  Per-class: ACK=0.852  BCC=0.839  MEL=0.500  NEV=0.838  SCC=0.358  SEK=0.915

  ── Per-class threshold tuning on val ensemble probs ──

  Threshold tuning  (baseline macro F1 = 0.7231

In [ ]:
test_preds_final = np.argmax(avg_test_probs / tuned_thresholds, axis=1)

n = CONFIG["num_classes"]
cls = CONFIG["class_names"]
acc  = accuracy_score(test_labels_np, test_preds_final)
bal  = balanced_accuracy_score(test_labels_np, test_preds_final)
f1m  = f1_score(test_labels_np, test_preds_final, average="macro",    zero_division=0)
f1w  = f1_score(test_labels_np, test_preds_final, average="weighted", zero_division=0)
f1pc = f1_score(test_labels_np, test_preds_final, average=None,
                labels=list(range(n)), zero_division=0)

print("\n===== FINAL TEST RESULTS =====")

print(f"Ensemble: Top-{CONFIG['top_k']}" +
      (" + SWA" if swa_active else "") +
      " + 5-view TTA + Per-class Thresholds")

print("\nMetrics:")
print(f"Accuracy          : {acc:.4f} ({acc*100:.2f}%)")
print(f"Balanced Accuracy : {bal:.4f} ({bal*100:.2f}%)")
print(f"F1-Macro          : {f1m:.4f}")
print(f"F1-Weighted       : {f1w:.4f}")

print("\nPer-class F1:")
for c, v in zip(cls, f1pc):
    print(f"{c:5s} : {v:.4f}")

print("\n" + classification_report(
    test_labels_np, test_preds_final, target_names=cls, zero_division=0))

cm = confusion_matrix(test_labels_np, test_preds_final)
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=cls, yticklabels=cls, ax=ax, linewidths=0.5)
ax.set(xlabel="Predicted", ylabel="True",
       title="Final Test — Confusion Matrix (Ensemble + Threshold Tuning)")
plt.tight_layout()
p = f"{CONFIG['save_dir']}confusion_matrix_final.png"
plt.savefig(p, dpi=150); plt.close(); print(f"  >> CM → {p}")

def plot_curves(hist, save_dir):
    ep = range(1, len(hist["train_acc"])+1)
    fig, axes = plt.subplots(2,2,figsize=(15,10))
    fig.suptitle("Training vs Validation", fontsize=15, fontweight="bold")
    pairs = [("Accuracy","train_acc","val_acc"),
             ("Balanced Acc","train_bal","val_bal"),
             ("F1-Macro","train_f1","val_f1"),
             ("Loss","train_loss","val_loss")]
    for ax,(t,tk,vk) in zip(axes.flat, pairs):
        ax.plot(ep, hist[tk], "#2196F3", lw=2, label="Train")
        ax.plot(ep, hist[vk], "#F44336", lw=2, ls="--", label="Val")
        ax.set_title(t); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    q = f"{save_dir}training_curves.png"; plt.savefig(q,dpi=150); plt.close()
    print(f"  >> Curves → {q}")

plot_curves(history, CONFIG["save_dir"])

def plot_roc(labels, probs, cls_names, save_dir):
    n = len(cls_names); ohe = np.eye(n)[labels]
    fig, ax = plt.subplots(figsize=(10,8))
    aucs = {}
    for i,(c,col) in enumerate(zip(cls_names, plt.cm.tab10(np.linspace(0,1,n)))):
        fpr,tpr,_ = roc_curve(ohe[:,i], probs[:,i])
        a = sk_auc(fpr,tpr); aucs[c]=a
        ax.plot(fpr,tpr,color=col,lw=2,label=f"{c}  AUC={a:.3f}")
    try: mac = roc_auc_score(ohe, probs, average="macro", multi_class="ovr")
    except: mac = np.mean(list(aucs.values()))
    ax.plot([0,1],[0,1],"k--",lw=1)
    ax.set(xlim=[0,1],ylim=[0,1.02],xlabel="FPR",ylabel="TPR",
           title=f"ROC Curves — Macro AUC={mac:.4f}")
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    plt.tight_layout()
    q = f"{save_dir}roc_test.png"; plt.savefig(q,dpi=150); plt.close()
    print(f"  >> ROC → {q}")
    return mac, aucs

mac_auc, cls_aucs = plot_roc(test_labels_np, avg_test_probs, cls, CONFIG["save_dir"])
#not used
colors = ["#4CAF50" if v>=0.75 else "#FF9800" if v>=0.55 else "#F44336" for v in f1pc]
fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(cls, f1pc, color=colors, edgecolor="white", lw=1.5)
for b,v in zip(bars,f1pc):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.015,
            f"{v:.3f}", ha="center", fontsize=12, fontweight="bold")
ax.axhline(np.mean(f1pc), color="navy", ls="--", lw=1.5,
           label=f"Macro={np.mean(f1pc):.3f}")
ax.set(ylim=[0,1.15], title="Per-Class F1 (Test, Ensemble+Tuned)", ylabel="F1 Score")
ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
q = f"{CONFIG['save_dir']}per_class_f1_test.png"; plt.savefig(q,dpi=150); plt.close()
print(f"  >> F1 bar → {q}")

results = {
    "method": f"Top-{CONFIG['top_k']} ensemble + SWA + 5-view TTA + threshold tuning",
    "best_train_epoch": int(best_epoch),
    "best_val_f1": round(float(best_val_f1), 4),
    "test_accuracy": round(float(acc), 4),
    "test_balanced_accuracy": round(float(bal), 4),
    "test_f1_macro": round(float(f1m), 4),
    "test_f1_weighted": round(float(f1w), 4),
    "test_macro_auc": round(float(mac_auc), 4),
    "per_class_f1": {c: round(float(v),4) for c,v in zip(cls,f1pc)},
    "per_class_auc": {c: round(float(a),4) for c,a in cls_aucs.items()},
    "tuned_thresholds": {c: round(float(t),4) for c,t in zip(cls, tuned_thresholds)},
}
with open(f"{CONFIG['save_dir']}final_results.json","w") as f:
    json.dump(results, f, indent=2)
print(f"\n  >> Results JSON → {CONFIG['save_dir']}final_results.json")


===== FINAL TEST RESULTS =====
Ensemble: Top-5 + SWA + 5-view TTA + Per-class Thresholds

Metrics:
Accuracy          : 0.8348 (83.48%)
Balanced Accuracy : 0.7788 (77.88%)
F1-Macro          : 0.8011
F1-Weighted       : 0.8278

Per-class F1:
ACK   : 0.8660
BCC   : 0.8630
MEL   : 0.8889
NEV   : 0.8800
SCC   : 0.4314
SEK   : 0.8772

              precision    recall  f1-score   support

         ACK       0.86      0.88      0.87        96
         BCC       0.83      0.89      0.86       141
         MEL       1.00      0.80      0.89         5
         NEV       0.82      0.94      0.88        35
         SCC       0.55      0.35      0.43        31
         SEK       0.96      0.81      0.88        31

    accuracy                           0.83       339
   macro avg       0.84      0.78      0.80       339
weighted avg       0.83      0.83      0.83       339

  >> CM → /kaggle/working/confusion_matrix_final.png
  >> Curves → /kaggle/working/training_curves.png
  >> ROC → /kaggle/wor